# Exploración y Análisis de Datos con yfinance (OHLCV)  
Este notebook descarga datos desde **Yahoo Finance** usando `yfinance` y realiza un análisis exploratorio completo con **tablas y gráficas**.

**Qué incluye**
- Descarga de datos (acciones/ETFs/índices) y guardado en CSV (opcional)
- Revisión de estructura, tipos, valores nulos, duplicados
- Estadísticos descriptivos
- Rendimientos (simple y log), volatilidad rolling
- Drawdowns y métricas básicas de riesgo
- Comparación multi-activo y correlaciones
- Análisis de volumen
- Detección visual de cambios de régimen (rolling stats)

> Recomendación TFM: usa este notebook como “base de exploración” y luego migra funciones a `src/`.


## 0) Setup

In [ ]:
# Si ejecutas en Colab, descomenta:
# !pip install -r requirements.txt

from pathlib import Path
import numpy as np
import pandas as pd
import yfinance as yf
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 120)

# Carpetas (ajusta si tu repo tiene otra estructura)
DATA_RAW = Path("../data/raw")
DATA_RAW.mkdir(parents=True, exist_ok=True)

print("OK - Librerías cargadas")


## 1) Configuración: tickers y rango temporal

In [ ]:
# Tickers típicos para TFM:
# SPY = S&P500 ETF, QQQ = Nasdaq100 ETF, ^VIX = volatilidad, ^GSPC = índice S&P500
TICKERS = ["SPY", "QQQ", "^GSPC", "^VIX"]
START = "2000-01-01"
END = None  # None = hasta hoy

INTERVAL = "1d"  # '1d', '1wk', '1mo' (intradía depende de limitaciones de Yahoo)

print("Tickers:", TICKERS)
print("Start:", START, "End:", END, "Interval:", INTERVAL)


## 2) Descarga de datos (OHLCV)

In [ ]:
# Descarga multi-ticker.
# group_by='ticker' devuelve columnas por ticker cuando hay múltiples símbolos.
raw = yf.download(
    tickers=TICKERS,
    start=START,
    end=END,
    interval=INTERVAL,
    group_by="ticker",
    auto_adjust=False,
    threads=True
)

raw.head()


### 2.1) Guardar CSV (opcional, recomendado para reproducibilidad)

In [ ]:
# Guarda un CSV por ticker en data/raw
def save_raw_per_ticker(raw_df: pd.DataFrame, tickers: list[str], out_dir: Path, interval: str) -> None:
    if isinstance(raw_df.columns, pd.MultiIndex):
        for t in tickers:
            if t in raw_df.columns.get_level_values(0):
                df_t = raw_df[t].copy()
                df_t.to_csv(out_dir / f"{t.replace('^','')}_{interval}_raw.csv")
    else:
        raw_df.to_csv(out_dir / f"{tickers[0].replace('^','')}_{interval}_raw.csv")

save_raw_per_ticker(raw, TICKERS, DATA_RAW, INTERVAL)
print("Guardado en:", DATA_RAW.resolve())


## 3) Normalización a formato “largo” (DataFrame estándar)

In [ ]:
def to_long_ohlcv(raw_df: pd.DataFrame) -> pd.DataFrame:
    """Convierte el DataFrame de yfinance a formato largo:
    columnas: [Date, Ticker, Open, High, Low, Close, Adj_Close, Volume]
    """
    if isinstance(raw_df.columns, pd.MultiIndex):
        out = (
            raw_df
            .stack(level=0, future_stack=True)  # mueve tickers a filas
            .rename_axis(index=["Date", "Ticker"])
            .reset_index()
        )
        out.columns = [c.replace(" ", "_") for c in out.columns]
        return out
    else:
        out = raw_df.copy().reset_index()
        out["Ticker"] = "SINGLE"
        out.columns = [c.replace(" ", "_") for c in out.columns]
        return out

df_long = to_long_ohlcv(raw)
df_long.head()


### 3.1) Vista rápida por ticker

In [ ]:
df_long.groupby('Ticker').head(3)

## 4) Calidad de datos: nulos, duplicados, rangos, tipos

In [ ]:
def data_quality_report(df: pd.DataFrame) -> pd.DataFrame:
    rep = pd.DataFrame({
        "dtype": df.dtypes.astype(str),
        "n_missing": df.isna().sum(),
        "pct_missing": (df.isna().mean() * 100).round(2),
        "n_unique": df.nunique(dropna=True)
    })
    return rep.sort_values("pct_missing", ascending=False)

print("Filas:", len(df_long), "Columnas:", df_long.shape[1])
print("Duplicados (filas completas):", df_long.duplicated().sum())

quality = data_quality_report(df_long)
quality


## 5) Estadísticos descriptivos por ticker

In [ ]:
numeric_cols = [c for c in df_long.columns if c not in ["Date", "Ticker"]]

desc = (
    df_long
    .groupby("Ticker")[numeric_cols]
    .describe(percentiles=[0.01, 0.05, 0.5, 0.95, 0.99])
)

desc


## 6) Visualización: precio y volumen

In [ ]:
def plot_close_prices(raw_df: pd.DataFrame, tickers: list[str], title: str = "Close Price"):
    plt.figure()
    if isinstance(raw_df.columns, pd.MultiIndex):
        for t in tickers:
            if (t, "Close") in raw_df.columns:
                plt.plot(raw_df.index, raw_df[(t, "Close")], label=t)
    else:
        plt.plot(raw_df.index, raw_df["Close"], label=tickers[0])
    plt.title(title)
    plt.xlabel("Date")
    plt.ylabel("Price")
    plt.legend()
    plt.tight_layout()
    plt.show()

plot_close_prices(raw, TICKERS, "Precio de Cierre (Close)")


In [ ]:
def plot_volume(raw_df: pd.DataFrame, ticker: str):
    plt.figure()
    if isinstance(raw_df.columns, pd.MultiIndex):
        vol = raw_df[(ticker, "Volume")]
        plt.plot(raw_df.index, vol)
        plt.title(f"Volumen - {ticker}")
    else:
        plt.plot(raw_df.index, raw_df["Volume"])
        plt.title(f"Volumen - {ticker}")
    plt.xlabel("Date")
    plt.ylabel("Volume")
    plt.tight_layout()
    plt.show()

plot_volume(raw, "SPY")


## 7) Rendimientos y volatilidad (base para HMM / Change Point / LSTM)

In [ ]:
def get_close_wide(raw_df: pd.DataFrame, tickers: list[str]) -> pd.DataFrame:
    if isinstance(raw_df.columns, pd.MultiIndex):
        close = pd.DataFrame({t: raw_df[(t, "Close")] for t in tickers if (t, "Close") in raw_df.columns})
    else:
        close = raw_df[["Close"]].rename(columns={"Close": tickers[0]})
    return close

close = get_close_wide(raw, TICKERS)
close.tail()


In [ ]:
# Rendimientos
ret = close.pct_change()
logret = np.log(close).diff()

# Volatilidad rolling (anualizada aproximada para daily)
ROLL = 21  # ~1 mes bursátil
vol = logret.rolling(ROLL).std() * np.sqrt(252)

ret.describe().T


In [ ]:
# Gráfica de log-returns (una serie)
ticker_focus = "SPY"
plt.figure()
plt.plot(logret.index, logret[ticker_focus])
plt.title(f"Log-returns diarios - {ticker_focus}")
plt.xlabel("Date")
plt.ylabel("log-return")
plt.tight_layout()
plt.show()


In [ ]:
# Histograma de returns + momentos
plt.figure()
vals = logret[ticker_focus].dropna().values
plt.hist(vals, bins=80)
plt.title(f"Distribución de log-returns - {ticker_focus}")
plt.xlabel("log-return")
plt.ylabel("Frecuencia")
plt.tight_layout()
plt.show()

moments = pd.Series({
    "mean": np.mean(vals),
    "std": np.std(vals),
    "skew": pd.Series(vals).skew(),
    "kurtosis": pd.Series(vals).kurtosis()
})
moments


In [ ]:
# Volatilidad rolling
plt.figure()
plt.plot(vol.index, vol[ticker_focus])
plt.title(f"Volatilidad rolling {ROLL} días (anualizada) - {ticker_focus}")
plt.xlabel("Date")
plt.ylabel("Vol anualizada")
plt.tight_layout()
plt.show()


## 8) Drawdown (crisis 2008, COVID, etc.)

In [ ]:
def compute_drawdown(price: pd.Series) -> pd.DataFrame:
    cummax = price.cummax()
    dd = price / cummax - 1.0
    return pd.DataFrame({"price": price, "cummax": cummax, "drawdown": dd})

dd = compute_drawdown(close[ticker_focus].dropna())

plt.figure()
plt.plot(dd.index, dd["drawdown"])
plt.title(f"Drawdown - {ticker_focus}")
plt.xlabel("Date")
plt.ylabel("Drawdown")
plt.tight_layout()
plt.show()

dd["drawdown"].describe()


## 9) Comparación multi-activo: correlaciones

In [ ]:
corr = logret.dropna().corr()
corr


In [ ]:
# Heatmap simple con matplotlib (sin seaborn)
plt.figure()
plt.imshow(corr.values, aspect="auto")
plt.title("Correlación de log-returns (matriz)")
plt.xticks(range(len(corr.columns)), corr.columns, rotation=45, ha="right")
plt.yticks(range(len(corr.index)), corr.index)
plt.colorbar()
plt.tight_layout()
plt.show()


## 10) Señales simples de tendencia (MA crossover)

In [ ]:
SHORT = 50
LONG = 200

ma_s = close[ticker_focus].rolling(SHORT).mean()
ma_l = close[ticker_focus].rolling(LONG).mean()
signal = (ma_s > ma_l).astype(int)

plt.figure()
plt.plot(close.index, close[ticker_focus], label="Close")
plt.plot(ma_s.index, ma_s, label=f"MA{SHORT}")
plt.plot(ma_l.index, ma_l, label=f"MA{LONG}")
plt.title(f"{ticker_focus}: Close + medias móviles (tendencia)")
plt.xlabel("Date")
plt.ylabel("Price")
plt.legend()
plt.tight_layout()
plt.show()

signal.value_counts()


## 11) Tabla resumen final (por ticker)

In [ ]:
summary_rows = []
for t in close.columns:
    s = close[t].dropna()
    if len(s) < 300:
        continue
    lr = np.log(s).diff().dropna()
    ann_ret = lr.mean() * 252
    ann_vol = lr.std() * np.sqrt(252)
    max_dd = compute_drawdown(s)["drawdown"].min()
    summary_rows.append({
        "Ticker": t,
        "start": s.index.min().date(),
        "end": s.index.max().date(),
        "n_obs": len(s),
        "ann_log_return": float(ann_ret),
        "ann_vol": float(ann_vol),
        "max_drawdown": float(max_dd),
    })

summary = pd.DataFrame(summary_rows).set_index("Ticker").sort_index()
summary


## 12) Siguientes pasos (para tu TFM)
- Guardar `close`, `logret`, `vol` en `data/processed/`
- Implementar **HMM** sobre `logret['SPY']` y contrastar estados con crisis
- Implementar **Change Point Detection** sobre `logret` o `vol`
- Más adelante: integrar macro (FRED) y sentimiento (FinBERT)
